Pretrained vivit import

In [ ]:
# torch 환경 세팅 버전(기록용)
# !pip install -U torch==2.3.1
# !pip install torchvision==0.17.2

## GPU 설정 단계

In [1]:
!nvidia-smi

Tue Mar  4 05:32:02 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A6000    On   | 00000000:1A:00.0 Off |                    0 |
| 31%   39C    P2   172W / 300W |  37209MiB / 46068MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A6000    On   | 00000000:1B:00.0 Off |                    0 |
| 30%   

In [2]:
import torch
import os
# print(torch.cuda. is_available())|

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= "1"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
print('Device:', device)  # 출력결과: cuda 
print('Count of using GPUs:', torch.cuda.device_count())  
print('Current cuda device:', torch.cuda.current_device()) 

Device: cuda
Count of using GPUs: 1
Current cuda device: 0


In [4]:
from transformers import AutoImageProcessor, VideoMAEForVideoClassification
import torch

model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base")

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
from transformers import VideoMAEConfig, VideoMAEModel

# Initializing a VideoMAE videomae-base style configuration
configuration = VideoMAEConfig()

# Randomly initializing a model from the configuration
model = VideoMAEModel(configuration)

# Accessing the model configuration
configuration = model.config

In [ ]:
import torch.nn as nn

sub = model.cuda()
model = nn.DataParallel(sub).to(device)

## Data input 수정 코드
- 입력 데이터 준비 및 모델 예측 실행
- 데이터 전처리 파트
- PyAV decoder를 사용해서 영상 디코딩

### 수정한 내용들
- 여러 데이터를 넣을 수 있게 진행
- def문은 유지
- for 말고 map 함수 쓰는 게 효율이 높아 보임

In [6]:
import av
import os
import numpy as np
from transformers import AutoImageProcessor, VideoMAEForVideoClassification

np.random.seed(0)

def read_video_pyav(container, indices):
    '''
    Decode the video with PyAV decoder.
    Args:
        container (`av.container.input.InputContainer`): PyAV container.
        indices (`List[int]`): List of frame indices to decode.
    Returns:
        result (np.ndarray): np array of decoded frames of shape (num_frames, height, width, 3).
    '''
    frames = []
    container.seek(0)
    start_index = indices[0]
    end_index = indices[-1]
    for i, frame in enumerate(container.decode(video=0)):
        if i > end_index:
            break
        if i >= start_index and i in indices:
            frames.append(frame)
    return np.stack([x.to_ndarray(format="rgb24") for x in frames])


def sample_frame_indices(clip_len, frame_sample_rate, seg_len):
    '''
    Sample a given number of frame indices from the video.
    Args:
        clip_len (`int`): Total number of frames to sample.
        frame_sample_rate (`int`): Sample every n-th frame.
        seg_len (`int`): Maximum allowed index of sample's last frame.
    Returns:
        indices (`List[int]`): List of sampled frame indices
    '''
    converted_len = int(clip_len * frame_sample_rate)
    
    if seg_len == 0:
        raise ValueError("ERROR: 비디오의 총 프레임 수가 0입니다. 해당 비디오는 처리 불가")
    
    # seg_len이 너무 작을 경우 방지
    if seg_len < converted_len:
        # print(f"ERROR: `seg_len`({seg_len})이 `converted_len`({converted_len})보다 작아서 샘플링 불가")
        converted_len = seg_len
    
    # 샘플링할 프레임 범위 조정
    end_idx = np.random.randint(converted_len, seg_len) if seg_len > converted_len else seg_len
    start_idx = max(0, end_idx - converted_len)  # 음수가 되지 않도록 조정

    indices = np.linspace(start_idx, end_idx, num=clip_len)
    indices = np.clip(indices, start_idx, end_idx - 1).astype(np.int64)
    
    return indices

def process_video(video_file):
    container = av.open(video_file)
    
    # 비디오의 총 프레임 개수 가져오기
    total_frames = container.streams.video[0].frames
    if total_frames == 0:
        raise ValueError(f"ERROR: 비디오 {video_file}의 총 프레임 수가 0이므로 처리 불가")

    indices = sample_frame_indices(clip_len=16, frame_sample_rate=1, seg_len=total_frames)
    video = read_video_pyav(container=container, indices=indices)
    inputs = image_processor(list(video), return_tensors="pt")
    
    return inputs["pixel_values"] # 다른 모델들과 다르게, 픽셀 값으로 반환 # 기존 코드가 그래서 동일하게 변형함

# 비디오 파일이 저장된 디렉토리 경로
# video_folder_path = "/Data3/hj24/CCTV/DATA/video_data/train"
video_folder_path = "/Data3/hj24/CCTV/DATA/crashdata"
video_files = [os.path.join(video_folder_path, filename) for filename in os.listdir(video_folder_path) if filename.endswith(".mp4")]

image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base",)
model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base")

# map()을 사용하여 모든 비디오 파일을 처리
results = map(process_video, video_files)

# 결과 출력 # 생략 가능
for result in results:
    print(f"Output shape: {result}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Video classification
- json file을 사용해 label을 설정하려 했지만 실패
- data folder를 accident, non-accident로 분류해 label를 추가

In [7]:
import av
import numpy as np
import torch
import os
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, VideoMAEForVideoClassification

np.random.seed(0)
torch.manual_seed(0)

# 데이터셋 클래스 정의
class VideoDataset(Dataset):
    def __init__(self, video_files, labels, image_processor):
        self.video_files = video_files
        self.labels = labels
        self.image_processor = image_processor

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        container = av.open(self.video_files[idx])
        total_frames = container.streams.video[0].frames
        
        if total_frames == 0:
            raise ValueError(f"ERROR: 비디오 {self.video_files[idx]}의 프레임 수가 0")

        indices = sample_frame_indices(clip_len=16, frame_sample_rate=4, seg_len=total_frames)
        video = read_video_pyav(container, indices)
        inputs = self.image_processor(list(video), return_tensors="pt")
        
        return inputs["pixel_values"].squeeze(0), torch.tensor(self.labels[idx], dtype=torch.long)


# 데이터 분류
# accident, non-accident label 생성을 위해 데이터 폴더 분리 + 이걸 이용해 레이블 생성성
# video_folder_path = "/Data3/hj24/CCTV/DATA/video_data/train"
video_folder_path = "/Data3/hj24/CCTV/DATA/crashdata"
video_files = []
labels = []

accident_folder = os.path.join(video_folder_path, "accident")
if os.path.exists(accident_folder):
    for filename in os.listdir(accident_folder):
        if filename.endswith(".mp4"):
            video_files.append(os.path.join(accident_folder, filename))
            labels.append(1) 

non_accident_folder = os.path.join(video_folder_path, "non-accident")
if os.path.exists(non_accident_folder): 
    for filename in os.listdir(non_accident_folder):
        if filename.endswith(".mp4"):
            video_files.append(os.path.join(non_accident_folder, filename))
            labels.append(0) 

print(f"총 {len(video_files)}개의 비디오 파일이 로드됨.")
print(f"레이블 개수: {len(labels)}")
assert len(video_files) == len(labels), "❌ 데이터와 레이블 개수가 일치하지 않습니다!"


총 4500개의 비디오 파일이 로드됨.
레이블 개수: 4500


In [8]:
import torch
import av
import numpy as np
import torch
import os
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, VideoMAEForVideoClassification
from torchmetrics.classification import Accuracy

image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base")

# 새로운 분류 레이어 추가 # 분류 모델 수정 시 여기를 건드려서 진행하면 됨
class VideoClassifier(nn.Module):
    def __init__(self, base_model):
        super(VideoClassifier, self).__init__()
        self.videomae = base_model
        self.classifier = nn.Linear(self.videomae.config.hidden_size, 2)  # 2개의 클래스 (accident, non-accident)

    def forward(self, x):
        outputs = self.videomae(x)
        return outputs.logits if hasattr(outputs, "logits") else outputs # 로짓으로 변환

#     def forward(self, x):
#         return self.videomae(x).logits

model = VideoClassifier(model)

# 데이터로더 생성 # 데이터 로더 부분은 추가로 더 확인해봐야함
dataset = VideoDataset(video_files, labels, image_processor)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# 학습 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

model = model.to(device)
accuracy_metric = Accuracy(task="multiclass", num_classes=2).to(device)

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 학습

In [ ]:
# 학습 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

model = model.to(device)
accuracy_metric = Accuracy(task="multiclass", num_classes=2).to(device)

# 전체 모델 학습 가능하게 수정 # 여기를 freeze or peft ? package 지정해서 수정하기... 
for param in model.parameters():
    param.requires_grad = True

for epoch in range(10):
    accuracy_metric.reset()
    
    for inputs, labels in dataloader:  # 데이터 로딩
        # inputs = process_video(video_file)  
        inputs = inputs.to(device)
        
        # label = torch.tensor(label, dtype=torch.long, device=device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        
        # loss = criterion(outputs, labels)  
        optimizer.step()
        # loss.backward()
        
        preds = torch.argmax(outputs, dim=1)
        accuracy_metric.update(preds, labels)
    
    epoch_accuracy = accuracy_metric.compute().item()
    print(f"Epoch {epoch+1}, Accuracy: {epoch_accuracy:.4f}")

torch.save(model.state_dict(), "videomae_accident_classifier.pth")


Epoch 1, Loss: 0.5583555102348328, Accuracy: 0.6929
Epoch 2, Loss: 0.5440612435340881, Accuracy: 0.6929
Epoch 3, Loss: 0.5854769945144653, Accuracy: 0.6929
Epoch 4, Loss: 0.6987826228141785, Accuracy: 0.6929
Epoch 5, Loss: 0.6415257453918457, Accuracy: 0.6929
Epoch 6, Loss: 0.6579964756965637, Accuracy: 0.6929
Epoch 7, Loss: 0.5828778743743896, Accuracy: 0.6929
Epoch 8, Loss: 0.7490139603614807, Accuracy: 0.6929
Epoch 9, Loss: 0.555992603302002, Accuracy: 0.6929
Epoch 10, Loss: 0.5306158661842346, Accuracy: 0.6929


## 모델 평가

In [ ]:
# 모델 평가 부분

import torch
from torchmetrics.classification import Precision, Recall, F1Score

model.eval()

precision = Precision(task="multiclass", num_classes=2, average="macro").to(device)
recall = Recall(task="multiclass", num_classes=2, average="macro").to(device)
f1 = F1Score(task="multiclass", num_classes=2, average="macro").to(device)

# 평가 모드에서 `torch.no_grad()` 사용 (그래디언트 비활성화)
with torch.no_grad():
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        preds = torch.argmax(outputs, dim=1)  
        precision.update(preds, labels)
        recall.update(preds, labels)
        f1.update(preds, labels)

print(f" Precision: {precision.compute().to('cpu').item():.4f}")
print(f" Recall: {recall.compute().to('cpu').item():.4f}")
print(f" F1 Score: {f1.compute().to('cpu').item():.4f}")

precision.reset()
recall.reset()
f1.reset()


## Test set 추가해서 성능 평가 필요
- validation set을 추가해도 괜찮아 보임
- 하단: 학습 결과 시각화

In [ ]:
# test loader 생성해서 넣기
# test data 추가

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 테스트 예측값 저장
all_preds = []
all_labels = []

for inputs, labels in test_dataloader:
    inputs, labels = inputs.to(device), labels.to(device)

    with torch.no_grad():
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

    all_preds.extend(preds.cpu().numpy())  # GPU에서 CPU로 이동 후 저장
    all_labels.extend(labels.cpu().numpy())

# Confusion Matrix 계산
cm = confusion_matrix(all_labels, all_preds)

# Confusion Matrix 시각화
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Non-Accident", "Accident"], yticklabels=["Non-Accident", "Accident"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()
